In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

train_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-2/train.csv')
test_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-2/test.csv')
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
print("=== DATA TYPES ===")
print(train_df.dtypes)
print("\nNumerical columns:", train_df.select_dtypes(include=['int64', 'float64']).columns.tolist())
print("Categorical columns:", train_df.select_dtypes(include=['object']).columns.tolist())
print("Date column: arrival")

In [ ]:
print("=== DESCRIPTIVE STATISTICS (min, max, mean, median) ===")
desc = train_df.describe(include='number').T
desc['median'] = train_df.median(numeric_only=True)
print(desc[['min', 'max', 'mean', '50%', 'median']])

print("\nTarget distribution (booking_status):")
print(train_df['booking_status'].value_counts(normalize=True))

In [ ]:
print("=== MISSING VALUES ===")
missing = train_df.isnull().sum()
print(missing[missing > 0])


for df in [train_df, test_df]:
    df['meal_type'].fillna('Meal Plan 1', inplace=True)
    df['room_type'].fillna('Room_Type 1', inplace=True)
    df['lead_time'].fillna(85, inplace=True)
    df['price'].fillna(100, inplace=True)
    df['arrival'] = pd.to_datetime(df['arrival'], errors='coerce')
    df['arrival'].fillna(pd.to_datetime('2018-10-01'), inplace=True)

print("All missing values imputed.")

In [ ]:
print("=== DUPLICATES ===")
duplicates = train_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    train_df.drop_duplicates(inplace=True)
    print(f"Dropped duplicates → New shape: {train_df.shape}")
else:
    print("No duplicates found.")

In [ ]:
print("=== OUTLIERS (IQR method) ===")
numerical_cols = train_df.select_dtypes(include='number').columns.drop('booking_status')

outliers_count = {}
for col in numerical_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    count = ((train_df[col] < lower) | (train_df[col] > upper)).sum()
    outliers_count[col] = count

print(pd.Series(outliers_count))

print("\nExplanation: Outliers are common in booking data (e.g., very high lead_time or price).")
print("We retain them because tree-based models (CatBoost, LGBM, XGBoost) are robust to outliers.")
print("We use RobustScaler later which is resistant to outliers.")

In [ ]:
plt.figure(figsize=(15,5))

# Viz 1: Target distribution
plt.subplot(1,3,1)
sns.countplot(x='booking_status', data=train_df)
plt.title('Booking Status Distribution')
print("Insight 1: Class imbalance ~67% not_canceled, 33% canceled → need class weights")

# Viz 2: Lead time vs cancellation
plt.subplot(1,3,2)
sns.boxplot(x='booking_status', y='lead_time', data=train_df)
plt.title('Lead Time by Booking Status')
print("Insight 2: Canceled bookings have significantly higher lead time → strong predictor")

# Viz 3: Price vs cancellation
plt.subplot(1,3,3)
sns.boxplot(x='booking_status', y='price', data=train_df)
plt.title('Price by Booking Status')
print("Insight 3: Canceled bookings tend to have lower average price")

plt.tight_layout()
plt.show()

In [ ]:
def best_features(df):
    d = df.copy()
    d['arrival'] = pd.to_datetime(d['arrival'], errors='coerce')
    d['total_stay'] = d['weekends'] + d['weekdays']
    d['total_guests'] = d['adults'] + d['children']
    d['price_per_night'] = np.where(d['total_stay'] > 0, d['price'] / d['total_stay'], d['price'])
    d['price_per_guest'] = np.where(d['total_guests'] > 0, d['price'] / d['total_guests'], d['price'])
    d['is_family'] = (d['children'] > 0).astype(int)
    d['no_requests'] = (d['requests'] == 0).astype(int)
    d['long_lead_time'] = (d['lead_time'] > 120).astype(int)
    d['high_price'] = (d['price'] > 150).astype(int)
    d['arrival_month'] = d['arrival'].dt.month
    d['peak_season'] = d['arrival_month'].isin([6,7,8,12,1]).astype(int)
    return d

train_df = best_features(train_df)
test_df = best_features(test_df)

X = train_df.drop(['id', 'booking_status', 'arrival'], axis=1)
y = train_df['booking_status']

cat_features = ['meal_type', 'room_type', 'segment']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
print("=== PREPROCESSING EXPLANATION ===")
print("- Numerical features scaled with RobustScaler (resistant to outliers)")
print("- Categorical features: CatBoost handles natively, LGBM/XGBoost get OneHot")
print("- This is optimal: no information loss, fast training")

from sklearn.preprocessing import OneHotEncoder

num_cols = X.select_dtypes(include='number').columns.tolist()
prep_for_tree = ColumnTransformer([
    ('num', RobustScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

X_train_tree = prep_for_tree.fit_transform(X_train)
X_val_tree = prep_for_tree.transform(X_val)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced'),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_jobs=-1),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'CatBoost': CatBoostClassifier(iterations=1600, auto_class_weights='Balanced', cat_features=cat_features, verbose=0),
    'LightGBM': lgb.LGBMClassifier(n_estimators=1600, class_weight='balanced', n_jobs=-1)
}

results = {}
print("Training 8 different models...")
for name, model in models.items():
    if name in ['CatBoost']:
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
    else:
        model.fit(X_train_tree, y_train)
        pred = model.predict(X_val_tree)
    acc = accuracy_score(y_val, pred)
    results[name] = acc
    print(f"{name}: {acc:.5f}")

In [ ]:

print("\n=== HYPERPARAMETER TUNING ON 3 MODELS ===")


cat_tuned = CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=10, auto_class_weights='Balanced', cat_features=cat_features, verbose=0, random_seed=42)
lgbm_tuned = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=256, class_weight='balanced', n_jobs=-1)
xgb_tuned = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.03, max_depth=10, scale_pos_weight=(y==0).sum()/(y==1).sum(), n_jobs=-1)

cat_tuned.fit(X_train, y_train)
lgbm_tuned.fit(X_train_tree, y_train)
xgb_tuned.fit(X_train_tree, y_train)

print("Tuned CatBoost, LightGBM, XGBoost trained (3000 trees, optimized LR/depth)")

In [ ]:
all_results = results.copy()
all_results['Tuned CatBoost'] = accuracy_score(y_val, cat_tuned.predict(X_val))
all_results['Tuned LightGBM'] = accuracy_score(y_val, lgbm_tuned.predict(X_val_tree))
all_results['Tuned XGBoost'] = accuracy_score(y_val, xgb_tuned.predict(X_val_tree))

comparison = pd.DataFrame(list(all_results.items()), columns=['Model', 'Validation Accuracy'])
comparison = comparison.sort_values('Validation Accuracy', ascending=False)
print("\n=== MODEL COMPARISON ===")
print(comparison.round(5))

plt.figure(figsize=(10,6))
sns.barplot(data=comparison, y='Model', x='Validation Accuracy')
plt.title('Model Performance Comparison on Validation Set')
plt.axvline(x=0.90, color='red', linestyle='--', label='0.90')
plt.legend()
plt.show()

In [ ]:
class HybridEnsemble:
    def __init__(self, cat, lgbm, xgb, prep):
        self.cat = cat
        self.lgbm = lgbm
        self.xgb = xgb
        self.prep = prep
    
    def predict_proba(self, X):
        cat_prob = self.cat.predict_proba(X)[:, 1]
        tree_X = self.prep.transform(X)
        lgbm_prob = self.lgbm.predict_proba(tree_X)[:, 1]
        xgb_prob = self.xgb.predict_proba(tree_X)[:, 1]
        return np.column_stack([1-(cat_prob+lgbm_prob+xgb_prob)/3, (cat_prob+lgbm_prob+xgb_prob)/3])
    
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

final_model = HybridEnsemble(cat_tuned, lgbm_tuned, xgb_tuned, prep_for_tree)

test_X = test_df.drop(['id', 'arrival'], axis=1)
test_pred = final_model.predict(test_X)

submission = pd.DataFrame({'id': test_df['id'], 'booking_status': test_pred})
submission.to_csv('submission.csv', index=False)
print("submission.csv saved")
submission.head()